# INICIO

In [ ]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import random
from IPython.display import Audio, display
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import soundfile as sf
from pathlib import Path
# Importar config
import sys
sys.path.append('../src')
from config import AUDIO_DIR, CSV_MONO, CSV_POLY, SAMPLE_RATE, N_FFT, HOP_LENGTH

In [ ]:
# Para máxima resolución usamos los parámetros originales del dataset (44.1kHz)
SR_ANALISIS = 44100  # resolución nativa de DataSED
N_FFT_ANALISIS = 4096
HOP_ANALISIS = 512

In [ ]:
# Cargar CSV para saber qué audios tienen eventos
df = pd.read_csv(CSV_MONO)
audios_unicos = df['sound_name'].unique().tolist()
print(f"Total de audios con anotaciones: {len(audios_unicos)}")

# FUNCIONES

In [ ]:
def analizar_duraciones(audio_dir):
    """
    Analiza las duraciones de todos los audios de una carpeta.
    Muestra estadísticas, el audio más corto y el más largo, y un histograma.
    
    Parámetros:
    -----------
    audio_dir : Path
        Carpeta donde están los archivos .wav
    """
    audios = sorted([f.name for f in Path(audio_dir).glob("*.wav")])
    print(f"Audios encontrados: {len(audios)}\n")

    duraciones = []
    for audio in audios:
        try:
            info = sf.info(Path(audio_dir) / audio)
            duraciones.append({
                'audio': audio,
                'duracion': info.duration,
                'sr': info.samplerate,
                'canales': info.channels
            })
        except Exception as e:
            print(f"Error leyendo {audio}: {e}")

    df_dur = pd.DataFrame(duraciones)

    # --- Estadísticas ---
    print("=" * 60)
    print("ESTADÍSTICAS DE DURACIÓN")
    print("=" * 60)
    print(df_dur['duracion'].describe())
    print(f"\nDuración mínima:     {df_dur['duracion'].min():.2f} s")
    print(f"Duración máxima:     {df_dur['duracion'].max():.2f} s")
    print(f"Desviación estándar: {df_dur['duracion'].std():.2f} s")
    print(f"¿Todas iguales?      {df_dur['duracion'].nunique() == 1}")

    # --- Audio más corto y más largo ---
    idx_min = df_dur['duracion'].idxmin()
    idx_max = df_dur['duracion'].idxmax()

    print("\n" + "=" * 60)
    print("AUDIO MÁS CORTO")
    print("=" * 60)
    print(f"Nombre:    {df_dur.loc[idx_min, 'audio']}")
    print(f"Duración:  {df_dur.loc[idx_min, 'duracion']:.2f} s")
    print(f"SR:        {df_dur.loc[idx_min, 'sr']} Hz")
    print(f"Canales:   {df_dur.loc[idx_min, 'canales']}")

    print("\n" + "=" * 60)
    print("AUDIO MÁS LARGO")
    print("=" * 60)
    print(f"Nombre:    {df_dur.loc[idx_max, 'audio']}")
    print(f"Duración:  {df_dur.loc[idx_max, 'duracion']:.2f} s "
          f"({df_dur.loc[idx_max, 'duracion']/60:.2f} min)")
    print(f"SR:        {df_dur.loc[idx_max, 'sr']} Hz")
    print(f"Canales:   {df_dur.loc[idx_max, 'canales']}")

    # --- Top 5 más cortos y más largos ---
    print("\n" + "=" * 60)
    print("TOP 5 AUDIOS MÁS CORTOS")
    print("=" * 60)
    print(df_dur.nsmallest(5, 'duracion')[['audio', 'duracion']].to_string(index=False))

    print("\n" + "=" * 60)
    print("TOP 5 AUDIOS MÁS LARGOS")
    print("=" * 60)
    print(df_dur.nlargest(5, 'duracion')[['audio', 'duracion']].to_string(index=False))

    # --- Histograma ---
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.hist(df_dur['duracion'], bins=40, color='#4ECDC4', edgecolor='black')
    ax.axvline(df_dur['duracion'].median(), color='yellow', linestyle='--',
               label=f"Mediana: {df_dur['duracion'].median():.1f} s")
    ax.axvline(df_dur['duracion'].mean(), color='salmon', linestyle='--',
               label=f"Media: {df_dur['duracion'].mean():.1f} s")
    ax.set_xlabel("Duración (s)")
    ax.set_ylabel("Número de audios")
    ax.set_title("Distribución de duraciones de los audios")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return df_dur

In [ ]:
df_anotaciones = pd.read_csv(CSV_MONO)

_PALETA = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#A78BFA', '#F472B6',
           '#34D399', '#FB923C', '#60A5FA', '#F87171', '#C084FC',
           '#2DD4BF', '#FBBF24', '#818CF8', '#FCA5A5', '#6EE7B7']

def analizar_audio(audio_name, guardar=False):
    audio_path = AUDIO_DIR / audio_name
    y, sr = librosa.load(audio_path, sr=SR_ANALISIS, mono=True)
    duracion = len(y) / sr

    # ============================================================
    # FFT DEL AUDIO COMPLETO
    # ============================================================
    n_fft_completo = int(2 ** np.ceil(np.log2(len(y))))
    fft_completa = np.fft.rfft(y, n=n_fft_completo)
    freqs_fft = np.fft.rfftfreq(n_fft_completo, 1/sr)
    magnitud_fft = np.abs(fft_completa)
    potencia_db_fft = 10 * np.log10(magnitud_fft**2 + 1e-10)
    potencia_db_fft -= potencia_db_fft.max()   # normaliza a su propio máximo

    # ============================================================
    # STFT
    # ============================================================
    stft_complejo = librosa.stft(y, n_fft=N_FFT_ANALISIS, hop_length=HOP_ANALISIS)
    magnitud_stft = np.abs(stft_complejo)
    freqs_stft = librosa.fft_frequencies(sr=sr, n_fft=N_FFT_ANALISIS)
    tiempos = librosa.frames_to_time(np.arange(magnitud_stft.shape[1]), sr=sr, hop_length=HOP_ANALISIS)
    magnitud_db_stft = librosa.amplitude_to_db(magnitud_stft, ref=np.max)

    potencia_stft_prom = np.mean(magnitud_stft**2, axis=1)
    potencia_db_stft = 10 * np.log10(potencia_stft_prom + 1e-10)
    potencia_db_stft -= potencia_db_stft.max()  # normaliza a su propio máximo

    # ============================================================
    # ANOTACIONES
    # ============================================================
    df_audio = df_anotaciones[df_anotaciones['sound_name'] == audio_name].copy()

    # ============================================================
    # ESCALAS
    # ============================================================
    FREQ_MIN, FREQ_MAX = 20, 20000
    DB_MIN, DB_MAX = -80, 0

    cgraf = "#969696"

    plt.style.use('dark_background')
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f"Análisis de {audio_name} — {len(df_audio)} eventos "
        f"(SR={sr} Hz, dur={duracion:.1f}s)",
        fontsize=13, color='white'
    )

    # --- 1. Espectro de potencia FFT (eje X log 20 Hz–20 kHz) ---
    ax = axes[0, 0]
    mask_fft = (freqs_fft >= FREQ_MIN) & (freqs_fft <= FREQ_MAX)
    ax.semilogx(freqs_fft[mask_fft], potencia_db_fft[mask_fft],
                color=cgraf, linewidth=0.8)
    ax.set_xlim(FREQ_MIN, FREQ_MAX)
    ax.set_ylim(DB_MIN, DB_MAX)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Potencia (dB)")
    ax.set_title("Espectro de Potencia FFT")
    ax.grid(True, alpha=0.2, which='both')
    ax.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
    ax.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'])

    # --- 2. Espectro de potencia STFT (eje X log 20 Hz–20 kHz) ---
    ax = axes[0, 1]
    mask_stft = (freqs_stft >= FREQ_MIN) & (freqs_stft <= FREQ_MAX)
    ax.semilogx(freqs_stft[mask_stft], potencia_db_stft[mask_stft],
                color=cgraf, linewidth=0.8)
    ax.set_xlim(FREQ_MIN, FREQ_MAX)
    ax.set_ylim(DB_MIN, DB_MAX)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Potencia (dB)")
    ax.set_title("Espectro de Potencia STFT (promediado)")
    ax.grid(True, alpha=0.2, which='both')
    ax.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
    ax.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'])

    # --- 3. Espectrograma con etiquetas ---
    ax = axes[1, 0]
    img = ax.imshow(
        magnitud_db_stft, aspect='auto', origin='lower',
        extent=[tiempos[0], tiempos[-1], freqs_stft[0], freqs_stft[-1]],
        cmap='gray', vmin=DB_MIN, vmax=DB_MAX
    )
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Hz)")
    ax.set_title("Espectrograma de Magnitud (con etiquetas)")

    clases_unicas = df_audio['class_name'].unique().tolist()
    color_por_clase = {c: _PALETA[i % len(_PALETA)] for i, c in enumerate(clases_unicas)}

    for _, row in df_audio.iterrows():
        t_ini, t_fin = row['start_time'], row['end_time']
        clase = row['class_name']
        color = color_por_clase[clase]
        rect = patches.Rectangle(
            (t_ini, freqs_stft[0]),
            t_fin - t_ini, freqs_stft[-1] - freqs_stft[0],
            linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.15
        )
        ax.add_patch(rect)
        ax.text(
            (t_ini + t_fin) / 2, freqs_stft[-1] * 0.95,
            clase, ha='center', va='top', fontsize=7, color='white',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85, edgecolor='none')
        )

    # --- 4. Espectrograma sin etiquetas ---
    ax = axes[1, 1]
    ax.imshow(
        magnitud_db_stft, aspect='auto', origin='lower',
        extent=[tiempos[0], tiempos[-1], freqs_stft[0], freqs_stft[-1]],
        cmap='gray', vmin=DB_MIN, vmax=DB_MAX
    )
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Frecuencia (Hz)")
    ax.set_title("Espectrograma de Magnitud (sin etiquetas)")

    # Colorbar compartida
    cbar = fig.colorbar(img, ax=axes[1, :], fraction=0.025, pad=0.02)
    cbar.set_label("Magnitud (dB)", color='white')
    cbar.ax.yaxis.set_tick_params(color='white')

    fig.subplots_adjust(left=0.06, right=0.92, top=0.93, bottom=0.07,
                        wspace=0.18, hspace=0.28)

    if guardar:
        out_path = Path("results/figures") / f"analisis_{audio_name.replace('.wav', '.png')}"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=100, bbox_inches='tight', facecolor='black')
        print(f"Guardado: {out_path}")

    plt.show()
    return fig

In [ ]:
def escuchar_audio(audio_name, audio_dir=AUDIO_DIR, sr=None):
    audio_path = Path(audio_dir) / audio_name
    if not audio_path.exists():
        print(f"❌ No existe: {audio_path}")
        return
    
    # Cargar con librosa (permite resamplear si sr != None)
    y, sr_real = librosa.load(audio_path, sr=sr, mono=True)
    duracion = len(y) / sr_real
    
    print(f"🎧 {audio_name}")
    print(f"   Duración: {duracion:.2f} s")
    print(f"   Sample rate: {sr_real} Hz")
    print(f"   Muestras: {len(y)}")
    
    display(Audio(data=y, rate=sr_real))

# ANALISIS

In [ ]:
# Cargar ambos CSV
df_mono = pd.read_csv(CSV_MONO)
df_poly = pd.read_csv(CSV_POLY)

print("=" * 60)
print("MONOFÓNICO")
print("=" * 60)
print(f"Filas: {len(df_mono)}")
print(f"Columnas: {list(df_mono.columns)}")
print(f"Audios únicos: {df_mono['sound_name'].nunique()}")
print(f"Clases únicas: {df_mono['class_name'].nunique()}")
print()
print(df_mono.head(10))

print("\n" + "=" * 60)
print("POLIFÓNICO")
print("=" * 60)
print(f"Filas: {len(df_poly)}")
print(f"Audios únicos: {df_poly['sound_name'].nunique()}")
print(f"Clases únicas: {df_poly['class_name'].nunique()}")
print()
print(df_poly.head(10))

In [ ]:
df_duraciones = analizar_duraciones(AUDIO_DIR)

In [ ]:
# Evento de duración más larga
idx_max = df_mono['event_length'].idxmax()
evento_max = df_mono.loc[idx_max]

print("=" * 60)
print("EVENTO DE DURACIÓN MÁS LARGA")
print("=" * 60)
print(f"Audio:      {evento_max['sound_name']}")
print(f"Clase:      {evento_max['class_name']}")
print(f"Inicio:     {evento_max['start_time']:.2f} s")
print(f"Fin:        {evento_max['end_time']:.2f} s")
print(f"Duración:   {evento_max['event_length']:.2f} s "
      f"({evento_max['event_length']/60:.2f} min)")

# Top 10 eventos más largos
print("\n" + "=" * 60)
print("TOP 10 EVENTOS MÁS LARGOS")
print("=" * 60)
top10 = df_mono.nlargest(10, 'event_length')[
    ['sound_name', 'class_name', 'start_time', 'end_time', 'event_length']
]
print(top10.to_string(index=False))

# Estadísticas por clase de los eventos más largos
print("\n" + "=" * 60)
print("DURACIÓN MÁXIMA POR CLASE")
print("=" * 60)
max_por_clase = df_mono.groupby('class_name')['event_length'].max().sort_values(ascending=False)
print(max_por_clase)

# GRAFICAS

In [ ]:
fig = analizar_audio("S-0001.wav")
escuchar_audio("S-0001.wav")

In [ ]:
fig = analizar_audio("S-0003.wav")
escuchar_audio("S-0003.wav")

In [ ]:
fig = analizar_audio("S-0087.wav")
escuchar_audio("S-0087.wav")

In [ ]:
fig = analizar_audio("S-0088.wav")
escuchar_audio("S-0088.wav")

In [ ]:
fig = analizar_audio("S-0089.wav")
escuchar_audio("S-0089.wav")

In [ ]:
fig = analizar_audio("S-0221.wav")
escuchar_audio("S-0221.wav")

In [ ]:
fig = analizar_audio("S-0011.wav")
escuchar_audio("S-0011.wav")

In [ ]:
fig = analizar_audio("S-0554.wav")
escuchar_audio("S-0554.wav")

In [ ]:
fig = analizar_audio("S-0432.wav")
escuchar_audio("S-0432.wav")

In [ ]:
fig = analizar_audio("S-0402.wav")
escuchar_audio("S-0402.wav")

In [ ]:
fig = analizar_audio("S-0399.wav")
escuchar_audio("S-0399.wav")

In [ ]:
cadena = "S-0299.wav"
fig = analizar_audio(cadena)
escuchar_audio(cadena)

In [ ]:
cadena = "S-0079.wav"
fig = analizar_audio(cadena)
escuchar_audio(cadena)

In [ ]:
cadena = "S-0629.wav"
fig = analizar_audio(cadena)
escuchar_audio(cadena)

In [ ]:
cadena = "S-0189.wav"
fig = analizar_audio(cadena)
escuchar_audio(cadena)